In [ ]:
from scipy.optimize import curve_fit
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from scipy.spatial import distance_matrix
from scipy.spatial import distance
from scipy.optimize import minimize
from scipy.optimize import basinhopping
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
import math
import shutil
import time
from numba import jit
import os
import statistics

In [ ]:
##############################################
# To Create your platelet, you can vary the
# dimensions below. The rest of the data initialization
# section of the code does not need to be changed.
##############################################

z_dist = 5.44  # Vertical separation between the two layers within each platelet

flat_radius = 5.19  #flat plate radius

hex_radius = 5.19   #hexagon radius

# Relative lateral displacement between the two layers of a platelet
theta = 61 *np.pi/180   

n_rings = 3

x_dist = z_dist/np.tan(theta) #x distance shift of bottom layer based on theta
x_dist
y_dist=x_dist

In [ ]:
def hexa(n_rings,radius):
    #width and height of the hexagon
    w = radius*2/np.sqrt(3)
    h = radius
    N = int(n_rings/2)

    #generate empty arrays
    central_x = []
    central_y = []
    edge_x = []
    edge_y = []

    if (n_rings%2==1):
        print("n_rings must be even number")
    else:
        #top edge atom layer
        edge_x.append(0)
        edge_y.append(0)
        for i in range(N):
            edge_x.append(edge_x[-1]+1.5*w)
            edge_y.append(edge_y[-1])

        #first half of hexagon
        for i in range(N):
            central_x.append(edge_x[0]+0.75*w)
            central_y.append(edge_y[0]-0.5*h)
            for j in range(N -1 +i):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            central_x.append(edge_x[0])
            central_y.append(edge_y[0]-h)
            for j in range(N+i):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.insert(0,edge_x[0]-0.75*w)
            edge_y.insert(0,edge_y[0]-1.5*h)
            central_x.append(edge_x[0]+1.5*w)
            central_y.append(edge_y[0])
            for j in range(N -1 +i):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.append(central_x[-1]+1.5*w)
            edge_y.append(central_y[-1])

        #second half of hexagon
        for i in range(N-1):
            central_x.append(edge_x[0]+0.75*w)
            central_y.append(edge_y[0]-0.5*h)
            for j in range(N+(N-i)-1):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            central_x.append(edge_x[0]+1.5*w)
            central_y.append(edge_y[0]-h)
            for j in range(N+(N-i)-2):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.insert(0,edge_x[0]+0.75*w)
            edge_y.insert(0,edge_y[0]-1.5*h)
            central_x.append(edge_x[0]+1.5*w)
            central_y.append(edge_y[0])
            for j in range(N +(N-i)-3):
                central_x.append(central_x[-1]+1.5*w)
                central_y.append(central_y[-1])
            edge_x.append(central_x[-1]+1.5*w)
            edge_y.append(central_y[-1])
        central_x.append(edge_x[0]+0.75*w)
        central_y.append(edge_y[0]-0.5*h)
        for j in range(N):
            central_x.append(central_x[-1]+1.5*w)
            central_y.append(central_y[-1])
        central_x.append(edge_x[0]+1.5*w)
        central_y.append(edge_y[0]-h)
        for j in range(N-1):
            central_x.append(central_x[-1]+1.5*w)
            central_y.append(central_y[-1])
        edge_x.append(edge_x[0]+0.75*w)
        edge_y.append(edge_y[0]-1.5*h)
        for i in range(N):
            edge_x.append(edge_x[-1]+1.5*w)
            edge_y.append(edge_y[-1])

        #center of mass back to 0
        edge_x -= np.average(edge_x)
        edge_y -= np.average(edge_y)
        central_x -= np.average(central_x)
        central_y -= np.average(central_y)

        #NEW - MERGE EDGE AND CENTRAL
        central_x = np.concatenate((central_x,edge_x))
        central_y = np.concatenate((central_y,edge_y))

        #NEW - EDGE ATOMS
        edge_x = []
        edge_y = []

        edge_x.append(n_rings*0.75*w+0.5*w)
        edge_y.append(0)

        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]-h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]-0.75*w)
            edge_y.append(edge_y[-1]-1.5*h)
        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]-h)

        edge_x.append(edge_x[-1]-w)
        edge_y.append(edge_y[-1])
        for i in range(N-1):
            edge_x.append(edge_x[-1]-1.5*w)
            edge_y.append(edge_y[-1])
        edge_x.append(edge_x[-1]-w)
        edge_y.append(edge_y[-1])

        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]+h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]-0.75*w)
            edge_y.append(edge_y[-1]+1.5*h)
        edge_x.append(edge_x[-1]-0.5*w)
        edge_y.append(edge_y[-1]+h)

        edge_x.append(edge_x[-1]+0.5*w)
        edge_y.append(edge_y[-1]+h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]+0.75*w)
            edge_y.append(edge_y[-1]+1.5*h)
        edge_x.append(edge_x[-1]+0.5*w)
        edge_y.append(edge_y[-1]+h)

        edge_x.append(edge_x[-1]+w)
        edge_y.append(edge_y[-1])
        for i in range(N-1):
            edge_x.append(edge_x[-1]+1.5*w)
            edge_y.append(edge_y[-1])
        edge_x.append(edge_x[-1]+w)
        edge_y.append(edge_y[-1])

        edge_x.append(edge_x[-1]+0.5*w)
        edge_y.append(edge_y[-1]-h)
        for i in range(N-1):
            edge_x.append(edge_x[-1]+0.75*w)
            edge_y.append(edge_y[-1]-1.5*h)

    return central_x,central_y,edge_x,edge_y

In [ ]:

#create flat plate edge and central atoms for the small platelet
x,y,xx,yy = hexa(2,5.19)

#print total number of atoms
print("Total number of atoms, small platelet: ",len(x)+len(xx))
print("Long diameter, small platelet: %.3f A: " %(max(xx)-min(xx)))

z = np.zeros(len(x))
zz = np.zeros(len(xx))

#put x,y,z data into one array
CentralPlatelet = np.stack((x,y,z),axis=-1)
EdgePlatelet = np.stack((xx,yy,zz),axis=-1)

#make two platelets
CentralPlatelet2=CentralPlatelet.copy()
CentralPlatelet2[:,2]=CentralPlatelet[:,2]+z_dist
CentralPlatelet2[:,0]=CentralPlatelet[:,0]-x_dist

EdgePlatelet2=EdgePlatelet.copy()
EdgePlatelet2[:,2]=EdgePlatelet[:,2]+z_dist
EdgePlatelet2[:,0]=EdgePlatelet[:,0]-x_dist

#fit into one array
S_CentralPlate = np.vstack((CentralPlatelet,CentralPlatelet2))
S_EdgePlate = np.vstack((EdgePlatelet,EdgePlatelet2))


plt.figure(figsize=(7,7))
plt.plot(CentralPlatelet[:,0],CentralPlatelet[:,1],"o")
plt.plot(EdgePlatelet[:,0],EdgePlatelet[:,1],"o")
plt.grid()
plt.show()



# In-plane rotation used for the 30° testing configuration
theta = np.radians(30)
rotation_matrix = np.array([
    [np.cos(theta), -np.sin(theta)],
    [np.sin(theta),  np.cos(theta)]
])

# ---- Mirror and Shift Central Plate ----
S_CentralPlateMirr = S_CentralPlate.copy()

# Mirror in Z and shift
S_CentralPlateMirr[:, 2] = -S_CentralPlateMirr[:, 2] + z_dist

# Rotate only x and y
S_CentralPlateMirr[:, :2] = (rotation_matrix @ S_CentralPlateMirr[:, :2].T).T


# ---- Mirror and Shift Edge Plate ----
S_EdgePlateMirr = S_EdgePlate.copy()

S_EdgePlateMirr[:, 2] = -S_EdgePlateMirr[:, 2] + z_dist

S_EdgePlateMirr[:, :2] = (rotation_matrix @ S_EdgePlateMirr[:, :2].T).T

plt.scatter(S_EdgePlateMirr[:,0], S_EdgePlateMirr[:,1])
plt.scatter(S_CentralPlateMirr[:,0], S_CentralPlateMirr[:,1])



fig = plt.figure(figsize=(8,6))
ax = fig.add_subplot(projection='3d')
ax.scatter(S_CentralPlate[:,0],S_CentralPlate[:,1],S_CentralPlate[:,2]-10, color = 'red', label = "Central Original Platelet")
ax.scatter(S_EdgePlate[:,0],S_EdgePlate[:,1],S_EdgePlate[:,2]-10, color = 'red',label = "Edge Original Platelet")
ax.scatter(S_CentralPlateMirr[:,0],S_CentralPlateMirr[:,1],S_CentralPlateMirr[:,2]+10, color = 'blue',label = "Central Mirrored Platelet")
ax.scatter(S_EdgePlateMirr[:,0],S_EdgePlateMirr[:,1],S_EdgePlateMirr[:,2]+10, color = 'blue',label = "Edge Mirrored Platelet")
ax.set_xlim([-20, 20])
ax.set_ylim([-20, 20])
ax.set_zlim([-15, 15])
ax.set_xticks(list(range(-20,21,10)))
ax.set_xticklabels(list(range(-20,21,10)),fontsize=16)
ax.set_yticks(list(range(-20,21,10)))
ax.set_yticklabels(list(range(-20,21,10)),fontsize=16)
ax.set_zticks(list(range(-10,11,10)))
ax.set_zticklabels(list(range(-10,11,10)),fontsize=16)
ax.set_xlabel('X [Å]',labelpad=12,fontsize=18)
ax.set_ylabel('Y [Å]',labelpad=10,fontsize=18)
ax.set_zlabel('Z [Å]',labelpad=7,fontsize=18)
plt.show()



In [ ]:
from mpl_toolkits.mplot3d import Axes3D
import matplotlib.pyplot as plt

fig = plt.figure(figsize=(10, 8))
ax = fig.add_subplot(111, projection='3d')

# Plot original plates
ax.scatter(S_CentralPlate[:, 0], S_CentralPlate[:, 1], S_CentralPlate[:, 2]-10,
           color='crimson', s=25, alpha=0.8, label="Central Original")
ax.scatter(S_EdgePlate[:, 0], S_EdgePlate[:, 1], S_EdgePlate[:, 2]-10,
           color='darkorange', s=25, alpha=0.8, label="Edge Original")

# Plot mirrored plates
ax.scatter(S_CentralPlateMirr[:, 0], S_CentralPlateMirr[:, 1], S_CentralPlateMirr[:, 2]+10,
           color='navy', s=25, alpha=0.8, label="Central Mirrored")
ax.scatter(S_EdgePlateMirr[:, 0], S_EdgePlateMirr[:, 1], S_EdgePlateMirr[:, 2]+10,
           color='deepskyblue', s=25, alpha=0.8, label="Edge Mirrored")


ax.set_xlim([-20, 20])
ax.set_ylim([-20, 20])
ax.set_zlim([-15, 15])


ax.set_xlabel('X [Å]', fontsize=16, labelpad=12)
ax.set_ylabel('Y [Å]', fontsize=16, labelpad=12)
ax.set_zlabel('Z [Å]', fontsize=16, labelpad=12)


ax.tick_params(axis='both', which='major', labelsize=14)
ax.set_xticks(range(-20, 21, 10))
ax.set_yticks(range(-20, 21, 10))
ax.set_zticks(range(-15, 16, 10))


ax.legend(loc='upper left', fontsize=12)


ax.view_init(elev=30, azim=0)

plt.tight_layout()
plt.show()


In [ ]:
from pathlib import Path

DATA_DIR = Path("../data/testing/rotation_30deg")

f_Stack = np.genfromtxt(DATA_DIR / "dspacing_PMF.out")

sorted_indices = np.argsort(f_Stack[:, 0])
f_Stack = f_Stack[sorted_indices]
z_S = f_Stack[:, 0]


cen_dis = {}
ed_cen_dis = {}
ed_dis = {}
cen_ed_dis = {}


for i, spacing in enumerate(z_S):

    CentralPlateMirr = S_CentralPlateMirr.copy()
    CentralPlateMirr[:, 2] += spacing


    dist_matrix_cc = distance.cdist(CentralPlateMirr, S_CentralPlate)
    final_distances_cc = dist_matrix_cc[dist_matrix_cc <= 25]
    cen_dis[round(spacing, 5)] = final_distances_cc

    print('cc')
    print(
        f"Iteration {i}: d-spacing = {spacing}, "
        f"Kept {len(final_distances_cc)} distances"
    )

    ####

    EdgePlateMirr = S_EdgePlateMirr.copy()
    EdgePlateMirr[:, 2] += spacing

    dist_matrix_ec = distance.cdist(EdgePlateMirr, S_CentralPlate)
    final_distances_ec = dist_matrix_ec[dist_matrix_ec <= 25]
    ed_cen_dis[round(spacing, 5)] = final_distances_ec

    print('ec')
    print(
        f"Iteration {i}: d-spacing = {spacing}, "
        f"Kept {len(final_distances_ec)} distances"
    )

    ####

    CentralPlateMirr = S_CentralPlateMirr.copy()
    CentralPlateMirr[:, 2] += spacing

    dist_matrix_ce = distance.cdist(CentralPlateMirr, S_EdgePlate)
    final_distances_ce = dist_matrix_ce[dist_matrix_ce <= 25]
    cen_ed_dis[round(spacing, 5)] = final_distances_ce

    print('ce')
    print(
        f"Iteration {i}: d-spacing = {spacing}, "
        f"Kept {len(final_distances_ce)} distances"
    )

    ####

    EdgePlateMirr = S_EdgePlateMirr.copy()
    EdgePlateMirr[:, 2] += spacing

    dist_matrix_ee = distance.cdist(EdgePlateMirr, S_EdgePlate)
    final_distances_ee = dist_matrix_ee[dist_matrix_ee <= 25]
    ed_dis[round(spacing, 5)] = final_distances_ee

    print('ee')
    print(
        f"Iteration {i}: d-spacing = {spacing}, "
        f"Kept {len(final_distances_ee)} distances"
    )


# ------------------------------------------------------------------
# Save the pairwise-distance dictionaries
# ------------------------------------------------------------------

cc_file = DATA_DIR / "PairDis_1_CC.npy"
ec_file = DATA_DIR / "PairDis_1_EC.npy"
ce_file = DATA_DIR / "PairDis_1_CE.npy"
ee_file = DATA_DIR / "PairDis_1_EE.npy"

np.save(cc_file, cen_dis, allow_pickle=True)
np.save(ec_file, ed_cen_dis, allow_pickle=True)
np.save(ce_file, cen_ed_dis, allow_pickle=True)
np.save(ee_file, ed_dis, allow_pickle=True)

print("\nSaved files:")
print(cc_file)
print(ec_file)
print(ce_file)
print(ee_file)


# ------------------------------------------------------------------
# Reload files to verify that they were saved correctly
# ------------------------------------------------------------------

loaded_cc = np.load(cc_file, allow_pickle=True).item()
loaded_ec = np.load(ec_file, allow_pickle=True).item()
loaded_ce = np.load(ce_file, allow_pickle=True).item()
loaded_ee = np.load(ee_file, allow_pickle=True).item()


# ------------------------------------------------------------------
# Compare d-spacing values between PMF file and saved .npy files
# ------------------------------------------------------------------

d_spacing_out = {round(x, 5) for x in f_Stack[:, 0]}

print("\nCC verification")
d_spacing_npy = {round(x, 5) for x in loaded_cc.keys()}

missing_in_out = d_spacing_npy - d_spacing_out
missing_in_npy = d_spacing_out - d_spacing_npy

print(f"Missing in .out file: {missing_in_out}")
print(f"Missing in .npy file: {missing_in_npy}")

for key in list(loaded_cc.keys())[:5]:
    print(
        f"d-spacing: {key}, "
        f"Number of distances: {len(loaded_cc[key])}"
    )


print("\nEC verification")
d_spacing_npy = {round(x, 5) for x in loaded_ec.keys()}

missing_in_out = d_spacing_npy - d_spacing_out
missing_in_npy = d_spacing_out - d_spacing_npy

print(f"Missing in .out file: {missing_in_out}")
print(f"Missing in .npy file: {missing_in_npy}")

for key in list(loaded_ec.keys())[:5]:
    print(
        f"d-spacing: {key}, "
        f"Number of distances: {len(loaded_ec[key])}"
    )


print("\nCE verification")
d_spacing_npy = {round(x, 5) for x in loaded_ce.keys()}

missing_in_out = d_spacing_npy - d_spacing_out
missing_in_npy = d_spacing_out - d_spacing_npy

print(f"Missing in .out file: {missing_in_out}")
print(f"Missing in .npy file: {missing_in_npy}")

for key in list(loaded_ce.keys())[:5]:
    print(
        f"d-spacing: {key}, "
        f"Number of distances: {len(loaded_ce[key])}"
    )


print("\nEE verification")
d_spacing_npy = {round(x, 5) for x in loaded_ee.keys()}

missing_in_out = d_spacing_npy - d_spacing_out
missing_in_npy = d_spacing_out - d_spacing_npy

print(f"Missing in .out file: {missing_in_out}")
print(f"Missing in .npy file: {missing_in_npy}")

for key in list(loaded_ee.keys())[:5]:
    print(
        f"d-spacing: {key}, "
        f"Number of distances: {len(loaded_ee[key])}"
    )